# MCP tools, pantry resources and client integration

The brief gives two possible capabilities: a deterministic calculation tool and a dynamic pantry resource. They are complementary rather than interchangeable. A **tool** accepts arguments and performs work, for example serving scaling. A **resource** exposes information under a URI, for example a pantry inventory. An application can use both. The pantry below is a synthetic demonstration stored in pantry_example.json, not the user’s real pantry. The server reads it on every resource request, so changes to that file appear on the next read.

The brief also suggests `FastMCP` and `langchain-mcp-adapters`. They are at different layers: FastMCP defines and serves capabilities; the adapter lets a LangChain/LangGraph application discover and call MCP tools. The direct MCP client is an alternative client path, not a replacement for the server. This notebook starts a separate temporary server process, reads its resource, calls its tool directly, then discovers and calls that same tool through the adapter. No LLM or Groq call is involved.

Both approaches require validation and authorization in a real application. This example persists data in a local JSON file, but it has no write API, access controls or concurrency handling. Edit pantry_example.json and rerun the read to see the updated resource. Run from this folder using the project environment and install the optional comparison requirements first.


In [ ]:
import asyncio, json, sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.client import MultiServerMCPClient

HERE=Path.cwd().resolve()
if HERE.name != 'comparisons_v1': raise RuntimeError('Start this notebook in v2/comparisons_v1')
server=str((HERE/'pantry_mcp_server.py').resolve())

async def compare_mcp():
    params=StdioServerParameters(command=sys.executable,args=[server],cwd=str(HERE))
    async with stdio_client(params) as (reader,writer):
        async with ClientSession(reader,writer) as session:
            await session.initialize()
            resources=await session.list_resources()
            resource=await session.read_resource('pantry://inventory')
            direct=await session.call_tool('scale_amount',arguments={'amount':200,'from_servings':2,'to_servings':5})
            if direct.isError: raise RuntimeError(str(direct.content))
            result={'resource_names':[r.name for r in resources.resources],
                    'resource_content':[getattr(c,'text','') for c in resource.contents],
                    'direct_tool_result':direct.structuredContent or [getattr(c,'text','') for c in direct.content]}
    client=MultiServerMCPClient({'example':{'command':sys.executable,'args':[server], 'transport':'stdio', 'cwd':str(HERE)}})
    tools=await client.get_tools()
    tool=next(t for t in tools if t.name=='scale_amount')
    result['adapter_tools']=[t.name for t in tools]
    result['adapter_tool_result']=await tool.ainvoke({'amount':200,'from_servings':2,'to_servings':5})
    return result


Jupyter already runs an event loop. Running the MCP subprocess in a separate thread gives it its own event loop, which also avoids common Windows notebook subprocess limitations.


In [ ]:
result = await asyncio.to_thread(lambda: asyncio.run(compare_mcp()))
print(json.dumps(result, indent=2, ensure_ascii=False, default=str))


## Compare the approaches

| Aspect | MCP tool | MCP resource |
|---|---|---|
| Example | Scale 200 g from two to five portions | Read pantry inventory |
| Interaction | Call with validated arguments and receive a result | Read a URI and receive resource contents |
| Typical controller | The model can propose a tool call; the host validates and executes it | The application chooses what to load into context |
| Reproducibility | Arithmetic is deterministic for the same inputs | Content can change as the inventory changes |
| In the existing MVP | Serving scaling and unit conversion | Not connected; this notebook is an isolated example |

Both paths in the code above reach the same server. The LangChain adapter handles tool conversion/discovery; it does not supply security, state management, correct model decisions or a pantry database automatically.
